In [ ]:
from pathlib import Path
from collections import Counter
import importlib
import os
import sys
from statistics import mean

import pandas as pd
from IPython.display import display

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "team_a/configs/evaluation.json").is_file())
sys.path.insert(0, str(repo))
from team_a.src.common_runtime import agent_import_path, load_config, sha256, verify_decks, verify_sdk
from team_a.src.models.jiayi.model_1 import Model as Linear
from team_a.src.models.jiayi.model_2 import Model as MLP
from team_a.src.models.jiayi.self_play import SHAPING_SCALE, play_battle

SDK_DIR = os.environ.get("PTCG_SDK_DIR")  # Or replace with the official sample_submission path.
if not SDK_DIR:
    print("Blocked: set PTCG_SDK_DIR to the official sample_submission directory containing cg/ and deck.csv.")
    print("No native battles executed.")

In [ ]:
if SDK_DIR:
    sdk_path = Path(SDK_DIR).expanduser().resolve()
    config = load_config()
    sdk_hashes = verify_sdk(sdk_path, config)
    deck_path = sdk_path / "deck.csv"
    deck = [int(line) for line in deck_path.read_text().splitlines() if line.strip()]
    if len(deck) != 60:
        raise ValueError("Expected the official 60-card sample deck")
    deck_id = {"deck_sha256": sha256(deck_path)}
    verify_decks(deck_id, deck_id, config)
    model_dir = repo / "team_a/src/models/jiayi"
    specs = [("A-J1", Linear, model_dir / "aj1_bc.json"),
             ("A-J2", MLP, model_dir / "aj2_bc.json")]
    epsilon = 0.1
    seeds = (61000, 61001, 61002, 61003)
    print("Python", sys.version.split()[0], "platform", sys.platform)
    print("Seeds", seeds, "epsilon", epsilon, "shaping scale", SHAPING_SCALE)
    print("Deck SHA-256", deck_id["deck_sha256"])
    print("SDK hashes", sdk_hashes)
    print("Checkpoint hashes", {name: sha256(path) for name, _, path in specs})
    battles, decisions = [], []
    with agent_import_path(sdk_path):
        for name in ("cg.game", "cg.sim"):
            module = importlib.import_module(name)
            expected = sdk_path / (name.replace(".", "/") + ".py")
            if Path(module.__file__).resolve() != expected:
                raise RuntimeError("Restart the kernel to load cg from the configured SDK directory")
        for name, loader, path in specs:
            model = loader(path)
            for index, seed in enumerate(seeds):
                result, trace = play_battle((model, model), deck, seed, epsilon, bool(index % 2))
                counts = Counter(d["source"] for d in trace)
                battles.append(dict(result, policy=name, learned=counts["learned"], fallback=counts["heuristic"]))
                decisions.extend(dict(d, policy=name, seed=seed) for d in trace)
    display(pd.DataFrame(battles).reindex(columns=["policy", "python_seed", "reverse_player", "status", "result_code",
                                                    "steps", "learned", "fallback", "terminal_returns", "shaped_returns",
                                                    "invalid_responses", "engine_rejections", "error_message"]))

In [ ]:
if SDK_DIR:
    comparison = []
    for name, _, _ in specs:
        completed = [d for d in decisions if d["policy"] == name and "terminal_return" in d]
        if not completed:
            continue
        for scheme in ("terminal", "shaped"):
            comparison.append({"policy": name, "scheme": scheme, "accepted_decisions": len(completed),
                               "nonzero_rewards": sum(d[f"{scheme}_reward"] != 0 for d in completed),
                               "mean_absolute_decision_reward": mean(abs(d[f"{scheme}_reward"]) for d in completed),
                               "mean_absolute_return_shift": mean(abs(d[f"{scheme}_return"] - d["terminal_return"]) for d in completed)})
    if comparison:
        display(pd.DataFrame(comparison))
        examples = [next((d for d in decisions if d["source"] == source and "terminal_return" in d), None)
                    for source in ("learned", "heuristic")]
        display(pd.DataFrame([d for d in examples if d is not None]))
    else:
        print("No valid completed battles to compare.")